## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
from load import load, load_files
from embed import embed
from chunk import split_small_chunks, split_big_chunks
from database import create_vectorstore
import torch
model_name = "microsoft/harrier-oss-v1-0.6b"

/home/ahaitota/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load from huggingface

In [3]:
documents = load("rungalileo/ragbench", "techqa", "test")

In [ ]:
small_chunks = split_small_chunks(documents, model_name)

In [6]:
big_chunks = split_big_chunks(documents, model_name)

In [7]:
print(small_chunks[10])
print("-------------------------------")
print(big_chunks[10])

page_content='ERROR DESCRIPTION
 *  Rational Developer for System z Software Analyzer hangs when
   analyzing a program with many copybooks.
   
   There are two scenarios:
   Common is that the default COBOL rules are selected. The use
   of custom rules is not significant.
   1) Start Software Analyzer while the copybooks were
      downloading - hang with the large number (e.g. 2147483647%)
      at the (bottom right) status area.
   2) Wait until all the status message are done (copybook'
-------------------------------
page_content='3) Change the productid to be WBI. 


4) After editing the .nifregistry file, the line should now look like: 
<product installrooturi=" file:///C:/Program%20Files/IBM/WebSphere/ESB/ [file:///C:/Program%20Files/IBM/WebSphere/ESB/]" 
lastvisited="2008-07-16 14:17:00-0400" productid="WBI" version="6.1.0.17"/> 

Your version of WBI may be different, but as long as it is a 6.1.0.x release, it is relative to this Technote. 


5) Install the WebSphere Transfo

In [8]:
small_vectors = embed(small_chunks, model_name=model_name, device="cuda")
print(len(small_vectors), "vectors of dim", len(small_vectors[0]))


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1521.41it/s]


9317 vectors of dim 1024


In [9]:
big_vectors = embed(big_chunks, model_name=model_name, device="cuda")
print(len(big_vectors), "vectors of dim", len(big_vectors[0]))

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2773.44it/s]


2239 vectors of dim 1024


In [10]:
# from embed import embed
# from chunk import split_with_parents
# from database import create_vectorstore, retrieve_parent_chunks

# big_chunks, small_chunks = split_with_parents(documents, model_name)

# # Embed separately
# vectors = embed(small_chunks)

# # Pass pre-computed vectors into Chroma (no re-embedding)
# vectorstore = create_vectorstore(small_chunks, vectors)

# # Query
# results = retrieve_parent_chunks("your query", vectorstore, big_chunks)

In [ ]:
vectorstore = create_vectorstore(big_chunks, big_vectors)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3058.62it/s]


## Make search

In [13]:
# Search directly — no parent lookup needed
results = vectorstore.similarity_search("How do I reset the appliance to factory defaults?", k=5)

In [18]:
print([page.page_content for page in results][0])

1- Device Manager
2- Network
3- System
4- Logout

<ESC>- Main Menu, <ENTER>- Refresh, <CTRL-L>- Event Log
> 1

------- Device Manager --------------------------------------------------------

1- Phase Management
2- Outlet Management
3- Power Supply Status

<ESC>- Back, <ENTER>- Refresh, <CTRL-L>- Event Log
> 2

------- Outlet Management -----------------------------------------------------

1- Outlet Control/Configuration
2- Outlet Restriction

<ESC>- Back, <ENTER>- Refresh, <CTRL-L>- Event Log
> 1

------- Outlet Control/Configuration ------------------------------------------

1- SPA14 SPA-DE03 left ps ON
2- SPA14 SPA-DE04 left ps ON
3- NETSWFAB03A left ps OFF
4- NETSWFAB03B left ps OFF
5- unused5 ON
6- unused6 ON
7- unused7 ON
8- unused8 ON
9- Master Control/Configuration

<ESC>- Back, <ENTER>- Refresh, <CTRL-L>- Event Log
> 3

------- NZ80684-H1 ------------------------------------------------------------

Name : NETSWFAB03A left ps
Outlet : 3
State : OFF

1- Control Outlet
2- Conf

## Read the docx files directly

In [3]:

documents = load_files("datasets/applets")

ActivID Applet Framework ACA (Access Control Applet) Technical Specifications

Page 8



ActivID Applet Framework ACA (Access Control Applet) Technical Specifications

Page 8





















Confidential under NDA – Do Not Redistribute

Confidential under NDA – Do Not Redistribute







HID® ActivID® Applet Framework

ACA (Access Control Applet)
Full Card-Edge Technical Specifications



Document Reference: APPLET_4.0.ACA_TS_07.2022

Product Version: 4.0

		NOVEMBER 2024











hidglobal.com

hidglobal.com
















Copyright

© 2017-2024 HID Global Corporation/ASSA ABLOY AB. All rights reserved.

Trademarks

HID, HID Global, the HID Blue Brick logo, the Chain Design, ActivIdentity and ActivID are trademarks or registered trademarks of HID Global, ASSA ABLOY AB, or its affiliates(s) in the US and other countries and may not be used without permission. All other trademarks, service marks, and product or service names are trademarks or registered trademarks of their respe

In [5]:
big_chunks = split_big_chunks(documents, model_name)
big_vectors = embed(big_chunks, model_name=model_name, device="cuda")
print(len(big_vectors), "vectors of dim", len(big_vectors[0]))

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 3184.84it/s]


279 vectors of dim 1024


In [6]:
sum([len(chunk.page_content) for chunk in big_chunks])

469298

In [7]:
vectorstore = create_vectorstore(chunks=big_chunks, vectors=big_vectors, collection_name="applets")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2763.40it/s]


In [12]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import ContextConstructionConfig
import glob
from dotenv import load_dotenv
load_dotenv()

synthesizer = Synthesizer()
doc_paths = glob.glob('datasets/applets/*.docx')

context_config = ContextConstructionConfig(
    max_contexts_per_document=5,
    chunk_size=512,
    chunk_overlap=0,
    context_quality_threshold=0.5,
    context_similarity_threshold=0.5,
)

goldens = synthesizer.generate_goldens_from_docs(
    document_paths=doc_paths,
    include_expected_output=True,
    max_goldens_per_context=2,
    context_construction_config=context_config,
)

[Confident AI Synthesizer Log] SUCCESS: Successfully deleted: /tmp/deepeval_chroma_hpxaj67j

[Confident AI Synthesizer Log] SUCCESS: Context Construction: Utilizing 50 out of 256 chunks.

In [13]:
goldens

[Golden(input='What challenge length does GET CHALLENGE return for 3TDEA vs AES?', actual_output=None, expected_output='GET CHALLENGE returns an **8-byte challenge for 3TDEA** and a **16-byte challenge for AES**.', context=['AL AUTHENTICATE or RESET RETRY COUNTER (P1=0x01) command. If an APDU other than EXTERNAL AUTHENTICATE or RESET RETRY COUNTER (P1=0x01) is sent just after GET CHALLENGE APDU, the challenge is lost.\n\nThis command can be sent through contactless interface when contactless firewall is deactivated.\n\nCommand Message\n\nCoding for the GET CHALLENGE command message is as listed next.\n\nTable 14: Coding for GET CHALLENGE\n\nCLA\n\n80h\n\nINS\n\n84h\n\nP1\n\n00h\n\nP2\n\n01h (Other values are RFUs)\n\nLc\n\nEmpty\n\nData Field\n\nEmpty\n\nLe\n\n00h\n\n\n\nResponse Message\n\nData Field Returned in the Response Message\n\nThis command returns the challenge value coded on 8 bytes for the 3TDEA key or 16 bytes for AES key. This challenge is memorized inside the applet inst

In [14]:
dataframe = synthesizer.to_pandas()
dataframe.to_csv("datasets/applets_goldens.csv", index=False)
print(f"Generated {len(dataframe)} goldens")
dataframe.head()

Generated 40 goldens


,input,actual_output,expected_output,context,retrieval_context,n_chunks_per_context,context_length,evolutions,context_quality,synthetic_input_quality,source_file
0,What challenge length does GET CHALLENGE retur...,None,GET CHALLENGE returns an **8-byte challenge fo...,[AL AUTHENTICATE or RESET RETRY COUNTER (P1=0x...,None,3,4903,[Concretizing],None,0.60,datasets/applets/ActivID_ACA_Full_CardEdge_Spe...
1,If a GET CHALLENGE is followed by another APDU...,None,The GET CHALLENGE value is lost if any APDU ot...,[AL AUTHENTICATE or RESET RETRY COUNTER (P1=0x...,None,3,4903,[Hypothetical],None,0.95,datasets/applets/ActivID_ACA_Full_CardEdge_Spe...
2,"What status word returns if VERIFY P1=00h, key...",None,The status word returned is **6983h**.,"[ instance in this version, key reference 80h)...",None,3,5952,[Constrained],None,0.70,datasets/applets/ActivID_PIVExt_Full_CardEdge_...
3,How does contactless firewall enforcement affe...,None,If CHANGE REFERENCE DATA is sent over contactl...,"[ instance in this version, key reference 80h)...",None,3,5952,[In-Breadth],None,1.00,datasets/applets/ActivID_PIVExt_Full_CardEdge_...
4,Compare RESET RETRY COUNTER outcomes for corre...,None,- **Correct PUK:** The PIN retry counter is re...,[IV PIN not initialized or P2≠80h)\n\n9000h\n\...,None,3,5927,[Comparative],None,0.90,datasets/applets/ActivID_PIVExt_Full_CardEdge_...
